# 2 — Análise de Clientes de Alta Relevância

**Objetivo:** Segmentar clientes por comportamento e analisar como características de produto e atividade influenciam a retenção, gerando insights acionáveis para a equipe de negócios.

**Dataset:** `datasets/processed/bank_churn.db` (modelo dimensional SQLite)

**Premissas:**
- Dados provenientes do modelo dimensional já validado (dim_customers, dim_geography, fact_bank_churn)
- Segmentação baseada em idade, saldo, tenure e atividade
- Análise de retenção cruzada com `NumOfProducts` e `HasCrCard`
- Gráficos exclusivamente com Plotly (paleta BI Dark Green)

---
## 1. Setup e Conexão

## 1.1 Imports e constantes

In [1]:
# Imports e constantes de layout padrão BI Dark Green
import sqlite3
from pathlib import Path

import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Layout padrão — Paleta 1 (BI Dark Green)
LAYOUT_PADRAO: dict = dict(
    template='plotly_dark',
    paper_bgcolor='#1A1A1A',
    plot_bgcolor='#0F2015',
    font=dict(color='#CCCCCC', size=12, family='Arial'),
    title_font=dict(color='#39FF5A', size=16),
)

COLOR_SEQUENCE: list[str] = ['#39FF5A', '#004D54', '#2ECC71', '#CCCCCC']

# Cores para status churn
COLOR_MAP_CHURN: dict[str, str] = {'Retido': '#39FF5A', 'Churned': '#004D54'}

## 1.2 Conexão com o banco SQLite

In [2]:
# Conexão com o banco dimensional bank_churn.db
DB_PATH: Path = Path('..') / 'datasets' / 'processed' / 'bank_churn.db'
conn: sqlite3.Connection = sqlite3.connect(str(DB_PATH))

print(f'Conectado ao banco: {DB_PATH.name}')

Conectado ao banco: bank_churn.db


## 1.3 Carregar base analítica consolidada

In [3]:
# Query para construir a base analítica unificada a partir do modelo dimensional
SQL_BASE_ANALITICA: str = """
SELECT
    c.customer_id,
    c.surname,
    c.credit_score,
    c.gender,
    c.age,
    c.tenure,
    c.estimated_salary,
    g.geography_name                                         AS geography,
    f.balance,
    f.num_of_products,
    f.has_credit_card,
    f.is_active_member,
    f.exited,
    CASE f.exited WHEN 1 THEN 'Churned' ELSE 'Retido' END  AS status_churn
FROM fact_bank_churn f
INNER JOIN dim_customers  c ON f.customer_id  = c.customer_id
INNER JOIN dim_geography  g ON f.geography_id = g.geography_id
"""

df_bank_churn: pd.DataFrame = pd.read_sql_query(SQL_BASE_ANALITICA, conn)

print(f'Base analítica: {df_bank_churn.shape[0]:,} registros × {df_bank_churn.shape[1]} colunas')
df_bank_churn.head()

Base analítica: 10,000 registros × 14 colunas


,customer_id,surname,credit_score,gender,age,tenure,estimated_salary,geography,balance,num_of_products,has_credit_card,is_active_member,exited,status_churn
0,15565701,Ferri,698,Female,39,9,90212.38,Spain,161993.89,1,0,0,0,Retido
1,15565706,Akobundu,612,Male,35,1,83256.26,Spain,0.00,1,1,1,1,Churned
2,15565714,Cattaneo,601,Male,47,1,96517.97,France,64430.06,2,0,1,0,Retido
3,15565779,Kent,627,Female,30,6,188258.49,Germany,57809.32,1,1,0,0,Retido
4,15565796,Docherty,745,Male,48,10,74510.65,Germany,96048.55,1,1,0,0,Retido


---
## 2. Segmentação de Clientes por Comportamento

## 2.1 Definição dos segmentos

Critérios de segmentação baseados em **idade**, **saldo**, **tenure** e **atividade**:

| Segmento | Critério |
|---|---|
| Jovens com Alto Saldo | Age < 35 e Balance > mediana |
| Jovens com Baixo Saldo | Age < 35 e Balance ≤ mediana |
| Maduros Ativos | Age 35-50 e IsActiveMember = 1 |
| Maduros Inativos | Age 35-50 e IsActiveMember = 0 |
| Seniores Engajados | Age > 50 e Tenure ≥ 5 |
| Seniores Recentes | Age > 50 e Tenure < 5 |

In [4]:
# Criar segmentos comportamentais com base em idade, saldo, tenure e atividade

def assign_segment(row: pd.Series) -> str:
    """
    Classifica um cliente em um segmento comportamental.

    Args:
        row: Linha do DataFrame com colunas age, balance, is_active_member, tenure.

    Returns:
        Nome do segmento.
    """
    age: int = row['age']
    balance: float = row['balance']
    is_active: int = row['is_active_member']
    tenure: int = row['tenure']

    if age < 35:
        if balance > balance_mediana:
            return 'Jovens Alto Saldo'
        return 'Jovens Baixo Saldo'
    elif age <= 50:
        if is_active == 1:
            return 'Maduros Ativos'
        return 'Maduros Inativos'
    else:
        if tenure >= 5:
            return 'Seniores Engajados'
        return 'Seniores Recentes'


balance_mediana: float = df_bank_churn['balance'].median()
print(f'Mediana do saldo (corte para segmentação): €{balance_mediana:,.2f}')

df_bank_churn['segmento'] = df_bank_churn.apply(assign_segment, axis=1)

print(f'\nDistribuição dos segmentos:')
print(df_bank_churn['segmento'].value_counts().to_string())

Mediana do saldo (corte para segmentação): €97,198.54

Distribuição dos segmentos:
segmento
Maduros Inativos      2626
Maduros Ativos        2434
Jovens Baixo Saldo    1911
Jovens Alto Saldo     1768
Seniores Engajados     682
Seniores Recentes      579


## 2.2 Perfil de cada segmento

In [5]:
# Estatísticas agregadas por segmento
df_perfil_segmento: pd.DataFrame = df_bank_churn.groupby('segmento').agg(
    total_clientes=('customer_id', 'count'),
    idade_media=('age', 'mean'),
    saldo_medio=('balance', 'mean'),
    salario_medio=('estimated_salary', 'mean'),
    tenure_medio=('tenure', 'mean'),
    credit_score_medio=('credit_score', 'mean'),
    produtos_medio=('num_of_products', 'mean'),
    pct_ativos=('is_active_member', 'mean'),
    total_churned=('exited', 'sum'),
    taxa_churn=('exited', 'mean'),
).round(2)

df_perfil_segmento['pct_ativos'] = (df_perfil_segmento['pct_ativos'] * 100).round(1)
df_perfil_segmento['taxa_churn'] = (df_perfil_segmento['taxa_churn'] * 100).round(1)
df_perfil_segmento = df_perfil_segmento.sort_values('taxa_churn', ascending=False)

df_perfil_segmento

,total_clientes,idade_media,saldo_medio,salario_medio,tenure_medio,credit_score_medio,produtos_medio,pct_ativos,total_churned,taxa_churn
segmento,,,,,,,,,,
Seniores Recentes,579,59.42,78126.74,93458.56,2.21,650.04,1.47,66.0,273,47.0
Seniores Engajados,682,59.70,81500.45,100949.17,7.30,649.27,1.49,66.0,290,43.0
Maduros Inativos,2626,40.83,77401.66,101020.06,5.06,647.17,1.53,0.0,760,29.0
Maduros Ativos,2434,40.61,77663.00,99847.03,4.92,652.92,1.54,100.0,424,17.0
Jovens Alto Saldo,1768,29.33,131283.12,100653.14,5.01,651.01,1.37,51.0,190,11.0
Jovens Baixo Saldo,1911,29.40,20744.72,100304.27,5.10,652.25,1.70,51.0,100,5.0


## 2.3 Taxa de churn por segmento

In [6]:
# Gráfico de barras — taxa de churn por segmento comportamental
df_plot_segmento: pd.DataFrame = df_perfil_segmento.reset_index().sort_values('taxa_churn', ascending=True)

fig_churn_seg: go.Figure = px.bar(
    df_plot_segmento,
    x='taxa_churn',
    y='segmento',
    orientation='h',
    title='Taxa de Churn por Segmento Comportamental (%)',
    labels={'taxa_churn': 'Taxa de Churn (%)', 'segmento': 'Segmento'},
    color_discrete_sequence=['#39FF5A'],
    text='taxa_churn',
)
fig_churn_seg.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig_churn_seg.update_layout(**LAYOUT_PADRAO, xaxis=dict(range=[0, 65]))
fig_churn_seg.show()

## 2.4 Volume e composição dos segmentos

In [7]:
# Gráfico de barras empilhadas — volume de retidos vs churned por segmento
df_seg_status: pd.DataFrame = (
    df_bank_churn
    .groupby(['segmento', 'status_churn'])
    .agg(total=('customer_id', 'count'))
    .reset_index()
)

# Ordenar segmentos pelo volume total
ordem_segmentos: list[str] = (
    df_seg_status.groupby('segmento')['total']
    .sum()
    .sort_values(ascending=True)
    .index
    .tolist()
)

fig_vol_seg: go.Figure = px.bar(
    df_seg_status,
    x='total',
    y='segmento',
    color='status_churn',
    orientation='h',
    title='Volume de Clientes por Segmento — Retidos vs Churned',
    labels={'total': 'Nº de Clientes', 'segmento': 'Segmento', 'status_churn': 'Status'},
    color_discrete_map=COLOR_MAP_CHURN,
    category_orders={'segmento': ordem_segmentos},
    barmode='stack',
)
fig_vol_seg.update_layout(**LAYOUT_PADRAO)
fig_vol_seg.show()

## 2.5 Saldo médio e salário médio por segmento

In [8]:
# Gráfico de barras agrupadas — saldo médio e salário médio por segmento
df_financeiro: pd.DataFrame = df_perfil_segmento.reset_index()[['segmento', 'saldo_medio', 'salario_medio']]
df_financeiro = df_financeiro.sort_values('saldo_medio', ascending=True)

fig_financeiro: go.Figure = go.Figure()

fig_financeiro.add_trace(go.Bar(
    y=df_financeiro['segmento'],
    x=df_financeiro['saldo_medio'],
    name='Saldo Médio',
    orientation='h',
    marker_color='#39FF5A',
))

fig_financeiro.add_trace(go.Bar(
    y=df_financeiro['segmento'],
    x=df_financeiro['salario_medio'],
    name='Salário Médio',
    orientation='h',
    marker_color='#004D54',
))

fig_financeiro.update_layout(
    **LAYOUT_PADRAO,
    title='Perfil Financeiro por Segmento (€)',
    xaxis_title='Valor Médio (€)',
    barmode='group',
)
fig_financeiro.show()

### Insights — Segmentação

- **Seniores Recentes** (50+, tenure < 5) têm a **maior taxa de churn: 47%** — clientes mais velhos com relacionamento curto, altíssimo risco.
- **Seniores Engajados** (50+, tenure ≥ 5) também apresentam churn elevado: **43%** — tenure longo não garante retenção nessa faixa.
- **Maduros Inativos** (35-50, inativos) representam o **maior grupo de risco absoluto**: 760 churned em 2.626 clientes (29%).
- **Jovens Baixo Saldo** são os mais fiéis: **5.2% de churn** — menor risco, porém menor saldo (€20.7K médio).
- **Jovens Alto Saldo** combinam boa retenção (11%) com saldo alto (€131K médio) — **segmento premium com melhor relação risco-retorno**.

---
## 3. Comportamento — Influência de Produtos na Retenção

## 3.1 Taxa de churn por número de produtos

In [9]:
# Análise de churn por número de produtos contratados
df_churn_produtos: pd.DataFrame = (
    df_bank_churn
    .groupby('num_of_products')
    .agg(
        total_clientes=('customer_id', 'count'),
        total_churned=('exited', 'sum'),
        taxa_churn=('exited', 'mean'),
        saldo_medio=('balance', 'mean'),
        idade_media=('age', 'mean'),
    )
    .reset_index()
)
df_churn_produtos['taxa_churn_pct'] = (df_churn_produtos['taxa_churn'] * 100).round(2)
df_churn_produtos['taxa_retencao_pct'] = (100 - df_churn_produtos['taxa_churn_pct']).round(2)

print('📊 Churn por Número de Produtos')
print(df_churn_produtos[['num_of_products', 'total_clientes', 'total_churned', 'taxa_churn_pct', 'taxa_retencao_pct']].to_string(index=False))

📊 Churn por Número de Produtos
 num_of_products  total_clientes  total_churned  taxa_churn_pct  taxa_retencao_pct
               1            5084           1409           27.71              72.29
               2            4590            348            7.58              92.42
               3             266            220           82.71              17.29
               4              60             60          100.00               0.00


In [10]:
# Gráfico — taxa de churn e retenção por número de produtos
fig_produtos: go.Figure = go.Figure()

fig_produtos.add_trace(go.Bar(
    x=df_churn_produtos['num_of_products'].astype(str),
    y=df_churn_produtos['taxa_retencao_pct'],
    name='Retenção (%)',
    marker_color='#39FF5A',
    text=df_churn_produtos['taxa_retencao_pct'].apply(lambda x: f'{x:.1f}%'),
    textposition='inside',
))

fig_produtos.add_trace(go.Bar(
    x=df_churn_produtos['num_of_products'].astype(str),
    y=df_churn_produtos['taxa_churn_pct'],
    name='Churn (%)',
    marker_color='#004D54',
    text=df_churn_produtos['taxa_churn_pct'].apply(lambda x: f'{x:.1f}%'),
    textposition='inside',
))

fig_produtos.update_layout(
    **LAYOUT_PADRAO,
    title='Retenção vs Churn por Número de Produtos',
    xaxis_title='Nº de Produtos',
    yaxis_title='Percentual (%)',
    barmode='stack',
)
fig_produtos.show()

## 3.2 Perfil dos clientes por número de produtos

In [11]:
# Box plot — distribuição de idade por número de produtos e status de churn
fig_box_age_prod: go.Figure = px.box(
    df_bank_churn,
    x='num_of_products',
    y='age',
    color='status_churn',
    title='Distribuição de Idade por Nº de Produtos e Status de Churn',
    labels={'num_of_products': 'Nº de Produtos', 'age': 'Idade', 'status_churn': 'Status'},
    color_discrete_map=COLOR_MAP_CHURN,
)
fig_box_age_prod.update_layout(**LAYOUT_PADRAO)
fig_box_age_prod.show()

In [12]:
# Box plot — distribuição de saldo por número de produtos e status de churn
fig_box_bal_prod: go.Figure = px.box(
    df_bank_churn,
    x='num_of_products',
    y='balance',
    color='status_churn',
    title='Distribuição de Saldo por Nº de Produtos e Status de Churn',
    labels={'num_of_products': 'Nº de Produtos', 'balance': 'Saldo (€)', 'status_churn': 'Status'},
    color_discrete_map=COLOR_MAP_CHURN,
)
fig_box_bal_prod.update_layout(**LAYOUT_PADRAO)
fig_box_bal_prod.show()

### Insights — Produtos

- **2 produtos** é o ponto ótimo de retenção: apenas **7.6% de churn** (4.590 clientes).
- **1 produto** tem churn moderado de **27.7%** (5.084 clientes) — enorme potencial de cross-sell para 2 produtos.
- **3 produtos** dispara o churn para **82.7%** — evento crítico, sugere insatisfação ou canibalização.
- **4 produtos** = 100% de churn (60 clientes) — produto tóxico, requer investigação imediata.
- Clientes churned com 3+ produtos tendem a ser mais velhos e possuem saldos semelhantes aos retidos.

---
## 4. Comportamento — Influência do Cartão de Crédito na Retenção

## 4.1 Taxa de churn: cartão de crédito × membro ativo

In [13]:
# Análise cruzada: posse de cartão de crédito × status de membro ativo
df_bank_churn['label_cartao'] = df_bank_churn['has_credit_card'].map({1: 'Com Cartão', 0: 'Sem Cartão'})
df_bank_churn['label_ativo'] = df_bank_churn['is_active_member'].map({1: 'Ativo', 0: 'Inativo'})

df_churn_cartao_ativo: pd.DataFrame = (
    df_bank_churn
    .groupby(['label_cartao', 'label_ativo'])
    .agg(
        total_clientes=('customer_id', 'count'),
        total_churned=('exited', 'sum'),
        taxa_churn=('exited', 'mean'),
    )
    .reset_index()
)
df_churn_cartao_ativo['taxa_churn_pct'] = (df_churn_cartao_ativo['taxa_churn'] * 100).round(2)

print('📊 Churn — Cartão de Crédito × Membro Ativo')
print(df_churn_cartao_ativo[['label_cartao', 'label_ativo', 'total_clientes', 'total_churned', 'taxa_churn_pct']].to_string(index=False))

📊 Churn — Cartão de Crédito × Membro Ativo
label_cartao label_ativo  total_clientes  total_churned  taxa_churn_pct
  Com Cartão       Ativo            3607            482           13.36
  Com Cartão     Inativo            3448            942           27.32
  Sem Cartão       Ativo            1544            253           16.39
  Sem Cartão     Inativo            1401            360           25.70


In [14]:
# Gráfico de barras agrupadas — churn por combinação cartão × atividade
fig_cartao_ativo: go.Figure = px.bar(
    df_churn_cartao_ativo,
    x='label_cartao',
    y='taxa_churn_pct',
    color='label_ativo',
    barmode='group',
    title='Taxa de Churn — Cartão de Crédito × Status de Atividade',
    labels={'label_cartao': 'Posse de Cartão', 'taxa_churn_pct': 'Taxa de Churn (%)', 'label_ativo': 'Membro'},
    color_discrete_map={'Ativo': '#39FF5A', 'Inativo': '#004D54'},
    text='taxa_churn_pct',
)
fig_cartao_ativo.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig_cartao_ativo.update_layout(**LAYOUT_PADRAO, yaxis=dict(range=[0, 40]))
fig_cartao_ativo.show()

## 4.2 Análise cruzada: cartão × número de produtos

In [15]:
# Tabela cruzada — taxa de churn por cartão de crédito e número de produtos
df_cross_cartao_prod: pd.DataFrame = (
    df_bank_churn
    .groupby(['label_cartao', 'num_of_products'])
    .agg(
        total_clientes=('customer_id', 'count'),
        total_churned=('exited', 'sum'),
        taxa_churn=('exited', 'mean'),
    )
    .reset_index()
)
df_cross_cartao_prod['taxa_churn_pct'] = (df_cross_cartao_prod['taxa_churn'] * 100).round(2)

print('📊 Churn — Cartão × Nº Produtos')
print(df_cross_cartao_prod[['label_cartao', 'num_of_products', 'total_clientes', 'total_churned', 'taxa_churn_pct']].to_string(index=False))

📊 Churn — Cartão × Nº Produtos
label_cartao  num_of_products  total_clientes  total_churned  taxa_churn_pct
  Com Cartão                1            3578            991           27.70
  Com Cartão                2            3246            236            7.27
  Com Cartão                3             190            156           82.11
  Com Cartão                4              41             41          100.00
  Sem Cartão                1            1506            418           27.76
  Sem Cartão                2            1344            112            8.33
  Sem Cartão                3              76             64           84.21
  Sem Cartão                4              19             19          100.00


In [16]:
# Heatmap — taxa de churn por cartão de crédito × número de produtos
df_pivot_heatmap: pd.DataFrame = df_cross_cartao_prod.pivot(
    index='label_cartao',
    columns='num_of_products',
    values='taxa_churn_pct',
)

fig_heatmap: go.Figure = go.Figure(data=go.Heatmap(
    z=df_pivot_heatmap.values,
    x=[str(c) for c in df_pivot_heatmap.columns],
    y=df_pivot_heatmap.index.tolist(),
    text=df_pivot_heatmap.values.round(1),
    texttemplate='%{text:.1f}%',
    textfont=dict(size=14, color='white'),
    colorscale=[[0, '#0F2015'], [0.5, '#004D54'], [1, '#39FF5A']],
    colorbar=dict(title='Churn %'),
    hovertemplate='Cartão: %{y}<br>Produtos: %{x}<br>Churn: %{z:.1f}%<extra></extra>',
))

fig_heatmap.update_layout(
    **LAYOUT_PADRAO,
    title='Heatmap — Taxa de Churn: Cartão de Crédito × Nº de Produtos',
    xaxis_title='Nº de Produtos',
    yaxis_title='Posse de Cartão',
)
fig_heatmap.show()

### Insights — Cartão de Crédito

- **Posse de cartão não diferencia churn**: ~27.7% com cartão vs ~27.8% sem cartão para 1 produto — diferença irrelevante.
- O fator decisivo é **atividade do membro**: inativos têm ~26-27% churn vs ativos ~13-16%, independente do cartão.
- No heatmap, a combinação mais tóxica é **3-4 produtos** (82-100% churn) com ou sem cartão.
- **Cartão de crédito é neutro** como ferramenta de retenção — investimento em engajamento gera mais retorno.

---
## 5. Análise Cruzada — Segmento × Produtos

In [17]:
# Taxa de churn por segmento e número de produtos
df_seg_prod: pd.DataFrame = (
    df_bank_churn
    .groupby(['segmento', 'num_of_products'])
    .agg(
        total_clientes=('customer_id', 'count'),
        taxa_churn=('exited', 'mean'),
    )
    .reset_index()
)
df_seg_prod['taxa_churn_pct'] = (df_seg_prod['taxa_churn'] * 100).round(2)

# Filtrar apenas combinações com volume mínimo de 10 clientes para significância
df_seg_prod_filtrado: pd.DataFrame = df_seg_prod[df_seg_prod['total_clientes'] >= 10].copy()

fig_seg_prod: go.Figure = px.bar(
    df_seg_prod_filtrado,
    x='segmento',
    y='taxa_churn_pct',
    color='num_of_products',
    barmode='group',
    title='Taxa de Churn por Segmento × Nº de Produtos (mín. 10 clientes)',
    labels={
        'segmento': 'Segmento',
        'taxa_churn_pct': 'Taxa de Churn (%)',
        'num_of_products': 'Nº Produtos',
    },
    color_discrete_sequence=COLOR_SEQUENCE,
    text='taxa_churn_pct',
)
fig_seg_prod.update_traces(texttemplate='%{text:.0f}%', textposition='outside')
fig_seg_prod.update_layout(**LAYOUT_PADRAO, xaxis_tickangle=-25)
fig_seg_prod.show()

---
## 6. Ranking de Clientes de Alta Relevância

In [18]:
# Identificar clientes de alta relevância: alto saldo + alto salário + retidos + ativos
q75_balance: float = df_bank_churn['balance'].quantile(0.75)
q75_salary: float = df_bank_churn['estimated_salary'].quantile(0.75)

df_alta_relevancia: pd.DataFrame = df_bank_churn[
    (df_bank_churn['balance'] >= q75_balance)
    & (df_bank_churn['estimated_salary'] >= q75_salary)
    & (df_bank_churn['exited'] == 0)
    & (df_bank_churn['is_active_member'] == 1)
].copy()

print(f'Clientes de Alta Relevância (Q75+ saldo e salário, ativos, retidos): {len(df_alta_relevancia):,}')
print(f'  Saldo mínimo (Q75):   €{q75_balance:>12,.2f}')
print(f'  Salário mínimo (Q75): €{q75_salary:>12,.2f}')
print(f'\nDistribuição por segmento:')
print(df_alta_relevancia['segmento'].value_counts().to_string())
print(f'\nDistribuição por geografia:')
print(df_alta_relevancia['geography'].value_counts().to_string())

Clientes de Alta Relevância (Q75+ saldo e salário, ativos, retidos): 267
  Saldo mínimo (Q75):   €  127,644.24
  Salário mínimo (Q75): €  149,388.25

Distribuição por segmento:
segmento
Maduros Ativos        127
Jovens Alto Saldo     110
Seniores Engajados     20
Seniores Recentes      10

Distribuição por geografia:
geography
France     112
Germany     99
Spain       56


In [19]:
# Scatter plot — Clientes de alta relevância: saldo vs salário, colorido por segmento
fig_scatter: go.Figure = px.scatter(
    df_alta_relevancia,
    x='estimated_salary',
    y='balance',
    color='segmento',
    size='credit_score',
    hover_data=['customer_id', 'age', 'tenure', 'num_of_products', 'geography'],
    title=f'Clientes de Alta Relevância — Saldo vs Salário ({len(df_alta_relevancia)} clientes)',
    labels={
        'estimated_salary': 'Salário Estimado (€)',
        'balance': 'Saldo (€)',
        'segmento': 'Segmento',
        'credit_score': 'Credit Score',
    },
    color_discrete_sequence=COLOR_SEQUENCE,
)
fig_scatter.update_layout(**LAYOUT_PADRAO)
fig_scatter.show()

---
## 7. Risco de Churn — Clientes Valiosos em Perigo

In [20]:
# Identificar clientes valiosos que fizeram churn: alto saldo + alto salário + exited=1
df_churn_valiosos: pd.DataFrame = df_bank_churn[
    (df_bank_churn['balance'] >= q75_balance)
    & (df_bank_churn['estimated_salary'] >= q75_salary)
    & (df_bank_churn['exited'] == 1)
].copy()

total_valiosos: int = len(df_alta_relevancia) + len(df_churn_valiosos)
taxa_churn_valiosos: float = round(len(df_churn_valiosos) / total_valiosos * 100, 2) if total_valiosos > 0 else 0

print(f'🚨 Clientes Valiosos (Q75+ saldo e salário):')
print(f'   Total:           {total_valiosos:,}')
print(f'   Retidos (ativos): {len(df_alta_relevancia):,}')
print(f'   Churned:          {len(df_churn_valiosos):,}')
print(f'   Taxa de churn:    {taxa_churn_valiosos}%')
print(f'\nPerfil dos churned valiosos:')
print(f'   Idade média:      {df_churn_valiosos["age"].mean():.1f}')
print(f'   Tenure médio:     {df_churn_valiosos["tenure"].mean():.1f}')
print(f'   Produtos médio:   {df_churn_valiosos["num_of_products"].mean():.1f}')
print(f'   % com cartão:     {df_churn_valiosos["has_credit_card"].mean() * 100:.1f}%')
print(f'   % ativos:         {df_churn_valiosos["is_active_member"].mean() * 100:.1f}%')

🚨 Clientes Valiosos (Q75+ saldo e salário):
   Total:           429
   Retidos (ativos): 267
   Churned:          162
   Taxa de churn:    37.76%

Perfil dos churned valiosos:
   Idade média:      45.1
   Tenure médio:     5.0
   Produtos médio:   1.5
   % com cartão:     71.0%
   % ativos:         35.8%


In [21]:
# Comparação — distribuição de segmentos entre retidos valiosos vs churn valiosos
df_retidos_seg: pd.DataFrame = (
    df_alta_relevancia['segmento'].value_counts(normalize=True)
    .reset_index()
    .rename(columns={'proportion': 'pct'})
)
df_retidos_seg['grupo'] = 'Retidos Valiosos'
df_retidos_seg['pct'] = (df_retidos_seg['pct'] * 100).round(1)

df_churn_seg: pd.DataFrame = (
    df_churn_valiosos['segmento'].value_counts(normalize=True)
    .reset_index()
    .rename(columns={'proportion': 'pct'})
)
df_churn_seg['grupo'] = 'Churned Valiosos'
df_churn_seg['pct'] = (df_churn_seg['pct'] * 100).round(1)

df_comp_valiosos: pd.DataFrame = pd.concat([df_retidos_seg, df_churn_seg], ignore_index=True)

fig_comp: go.Figure = px.bar(
    df_comp_valiosos,
    x='segmento',
    y='pct',
    color='grupo',
    barmode='group',
    title='Composição de Segmentos — Retidos vs Churned Valiosos',
    labels={'segmento': 'Segmento', 'pct': 'Proporção (%)', 'grupo': 'Grupo'},
    color_discrete_map={'Retidos Valiosos': '#39FF5A', 'Churned Valiosos': '#004D54'},
    text='pct',
)
fig_comp.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig_comp.update_layout(**LAYOUT_PADRAO, xaxis_tickangle=-25)
fig_comp.show()

---
## 8. Encerramento

In [22]:
# Fechar conexão com o banco
conn.close()
print('✅ Conexão com o banco encerrada.')

✅ Conexão com o banco encerrada.


---
## 9. Diagnóstico Final — Clientes de Alta Relevância

### 9.1 Segmentação
- **Seniores** (50+) dominam o churn: **47% (Recentes)** e **43% (Engajados)** — faixa etária mais crítica.
- **Maduros Inativos** são o **maior grupo de risco absoluto**: 760 churned (29% de 2.626 clientes).
- **Jovens com Alto Saldo** são o **melhor perfil**: apenas 11% churn com saldo médio de €131K.
- **Jovens com Baixo Saldo** são os mais fiéis (5.2% churn) mas geram menor receita.

### 9.2 Produtos
- **2 produtos** é o ponto ótimo — **7.6% churn** (4.590 clientes, 92% de retenção).
- **1 produto** tem churn de **27.7%** — oportunidade massiva de cross-sell para 2 produtos (5.084 clientes).
- **3+ produtos** = churn catastrófico: **82.7%** (3 produtos) e **100%** (4 produtos).

### 9.3 Cartão de Crédito
- **Posse de cartão é neutra** para retenção — taxas praticamente idênticas com e sem cartão.
- **Atividade do membro é o fator decisivo**: inativos têm até o dobro do churn dos ativos.

### 9.4 Clientes Valiosos em Risco
- **429 clientes valiosos** (saldo e salário Q75+): **37.8% fizeram churn**.
- Churned valiosos: idade média 45.1 anos, apenas 35.8% eram ativos — perfil de **maduro/sênior inativo**.

### 9.5 Recomendações para Marina Costa (Gerente de Negócios)
1. **Cross-sell de 1→2 produtos**: prioridade nº 1 — reduz churn de 27.7% para 7.6% num pool de 5.084 clientes.
2. **Programa de reativação** para Maduros Inativos e Seniores — maior retorno em retenção.
3. **Alerta no 3º produto**: investigar causa raiz do churn de 83% — possível canibalização ou insatisfação.
4. **Engajamento > cartão de crédito**: campanhas de atividade superam oferta de cartão como ferramenta de retenção.
5. **Monitoramento de clientes valiosos inativos**: 37.8% de churn nesse grupo é inaceitável.